# TF-IDF Implementation from Scratch

**1. Library Imports**: Imports Pandas and NumPy for tabular and numerical array operations.

In [1]:
# Import pandas library for handling tabular data and dataframes
import pandas as pd

# Import numpy library for numerical computations and matrix operations
import numpy as np

**2. Sample Corpus**: Defines the list of sample text documents used for analysis.

In [2]:
# Define a sample corpus containing text documents (sentences) for analysis
corpus = [
    # Document 1: simple sentence about data science and python
    "Data science uses python",
    # Document 2: sentence describing python
    "Python is a language",
    # Document 3: sentence with repeating word 'science'
    "Data science is powerful science",
]

**3. Baseline Vocabulary Extraction**: Converts document strings to lowercase and computes document count and baseline sentence-level vocabulary size.

In [3]:
# Convert all document sentences in the corpus to lowercase for case-insensitivity
lower_case_corpus = [w.lower() for w in corpus]

# Find the set of unique lowercased sentences
vocab = set(lower_case_corpus)

# Count the total number of documents in the corpus
n_docs = len(corpus)

# Count the number of unique sentences (baseline vocabulary size)
n_words = len(vocab)

**4. Term Frequency (TF) Matrix**: Tokenizes documents into words, builds a sorted vocabulary mapping, constructs a raw count matrix, and normalizes by document length to compute TF.

In [4]:
# Initialize an empty list to store tokenized words for each document
tokenized_corpus = []

# Initialize a set to collect all unique vocabulary words across documents
temp_vocab_set = set()

# Iterate through each sentence in the corpus
for sentence in corpus:
    # Convert sentence to lowercase and split by whitespace into word tokens
    words = sentence.lower().split(' ')
    # Append the list of words for this document to tokenized_corpus
    tokenized_corpus.append(words)
    # Add unique words from this document to the vocabulary set
    temp_vocab_set.update(words)

# Sort unique vocabulary words alphabetically into a list
vocab = sorted(list(temp_vocab_set))

# Update n_words with the total number of unique vocabulary words
n_words = len(vocab)

# Create a dictionary mapping each word to a unique column index
word_to_idx = {word: idx for idx, word in enumerate(vocab)}

# Initialize a document-term count matrix of zeros with shape (n_docs, n_words)
counts_matrix = np.zeros((n_docs, n_words))

# Populate the count matrix with word occurrences per document
for i, doc_words in enumerate(tokenized_corpus):
    # Iterate through each word in the tokenized document
    for word in doc_words:
        # Check if the word exists in the vocabulary dictionary
        if word in word_to_idx:
            # Look up the column index for the word
            word_idx = word_to_idx[word]
            # Increment the term count for document i at word_idx
            counts_matrix[i, word_idx] += 1

# Calculate the length (number of words) of each document as a column vector
doc_lengths = np.array([len(doc_words) for doc_words in tokenized_corpus]).reshape(-1, 1)

# Prevent division by zero by setting any document length of 0 to 1
doc_lengths[doc_lengths == 0] = 1

# Calculate Term Frequency (TF): raw count divided by document length
tf_matrix = counts_matrix / doc_lengths

# Convert the TF numpy matrix to a pandas DataFrame with vocabulary as column names
df_tf = pd.DataFrame(tf_matrix, columns=vocab)

**5. Inverse Document Frequency (IDF) Vector**: Determines the document frequency of each vocabulary term and calculates logarithmic IDF weights.

In [5]:
# Compute Document Frequency (DF): count how many documents contain each word
docs_containing_word = np.sum(counts_matrix > 0, axis=0)

# Compute Inverse Document Frequency (IDF): log(total documents / document frequency)
idf_vector = np.log(n_docs / docs_containing_word)

**6. TF-IDF Matrix & Score Display**: Computes TF-IDF by multiplying TF dataframe columns by IDF weights and prints sorted scores for terms in each document.

In [6]:
# Calculate TF-IDF matrix by multiplying TF DataFrame columns by the IDF vector
df_tf_idf = df_tf.multiply(idf_vector, axis="columns")

# Print section header for TF-IDF scores
print("=== TF-IDF Scores per Document ===")

# Iterate over each document in the corpus
for i in range(n_docs):
    # Print document index and original sentence text
    print(f"\nDocument {i+1}: '{corpus[i]}'")
    
    # Extract the TF-IDF score series for the current document
    doc_series = df_tf_idf.iloc[i]
    
    # Select terms with non-negative scores
    active_words = doc_series[doc_series >= 0]
    
    # Sort terms by their TF-IDF scores in descending order
    sorted_words = active_words.sort_values(ascending=False)
    
    # Iterate through the sorted terms and their TF-IDF scores
    for word, score in sorted_words.items():
        # Check if the term actually appears in the current document
        if word in tokenized_corpus[i]:
            # Print the term and its formatted score (4 decimal places)
            print(f"  -> {word:<10} Score: {score:.4f}")

=== TF-IDF Scores per Document ===

Document 1: 'Data science uses python'
  -> uses       Score: 0.2747
  -> data       Score: 0.1014
  -> python     Score: 0.1014
  -> science    Score: 0.1014

Document 2: 'Python is a language'
  -> a          Score: 0.2747
  -> language   Score: 0.2747
  -> is         Score: 0.1014
  -> python     Score: 0.1014

Document 3: 'Data science is powerful science'
  -> powerful   Score: 0.2197
  -> science    Score: 0.1622
  -> data       Score: 0.0811
  -> is         Score: 0.0811
